# Aura AI — Qwen3 0.6B Korean QLoRA

Private Kaggle GPU fine-tuning run. Uses the existing curated Korean SFT train/validation splits, a pretrained Apache-2.0 Qwen3 base model, NF4 QLoRA adapters, validation checkpoints, and a generation smoke test.


In [ ]:
import os, json, torch
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen3-0.6B"
DATA_ROOT = Path("/kaggle/input/aura-ai-korean-sft-data")
OUTPUT = Path("/kaggle/working/aura-qwen3-0.6b-korean-qlora")
MAX_LENGTH = 768
SEED = 20261125

assert torch.cuda.is_available(), "Enable the Kaggle NVIDIA GPU accelerator."
print({"gpu": torch.cuda.get_device_name(0), "vram_gib": round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2)})
assert (DATA_ROOT / "train.jsonl").is_file() and (DATA_ROOT / "valid.jsonl").is_file(), "Korean SFT JSONL dataset is not mounted."

os.environ["TOKENIZERS_PARALLELISM"] = "false"
raw = load_dataset("json", data_files={
    "train": str(DATA_ROOT / "train.jsonl"),
    "validation": str(DATA_ROOT / "valid.jsonl")
})
print("DATASET_ROWS", {k: len(v) for k, v in raw.items()})

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def encode_row(row):
    messages = row.get("messages", [])
    messages = [m for m in messages if m.get("role") in {"system", "user", "assistant"} and isinstance(m.get("content"), str)]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)
    item = tokenizer(text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)
    item["labels"] = list(item["input_ids"])
    return item

tokenized = raw.map(encode_row, remove_columns=raw["train"].column_names, desc="Tokenizing Aura Korean conversations")
tokenized = tokenized.filter(lambda x: len(x["input_ids"]) >= 8)
print("TOKENIZED_ROWS", {k: len(v) for k, v in tokenized.items()})

class PadCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id
    def __call__(self, features):
        width = max(len(x["input_ids"]) for x in features)
        ids = torch.full((len(features), width), self.pad_id, dtype=torch.long)
        mask = torch.zeros((len(features), width), dtype=torch.long)
        labels = torch.full((len(features), width), -100, dtype=torch.long)
        for i, x in enumerate(features):
            n = len(x["input_ids"])
            ids[i, :n] = torch.tensor(x["input_ids"], dtype=torch.long)
            mask[i, :n] = 1
            labels[i, :n] = torch.tensor(x["labels"], dtype=torch.long)
        return {"input_ids": ids, "attention_mask": mask, "labels": labels}

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    torch_dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa"
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=str(OUTPUT),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1.5e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    eval_steps=300,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    group_by_length=True,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=PadCollator(tokenizer.pad_token_id),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)
result = trainer.train()
metrics = trainer.evaluate()
OUTPUT.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(OUTPUT, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT)
trainer.save_state()
(Path("/kaggle/working/aura-qwen3-training-metrics.json")).write_text(json.dumps({
    "base_model": BASE_MODEL,
    "train_rows": len(tokenized["train"]),
    "validation_rows": len(tokenized["validation"]),
    "train_metrics": result.metrics,
    "evaluation_metrics": metrics,
    "epochs": 2,
    "max_length": MAX_LENGTH,
    "method": "QLoRA NF4, rank 16"
}, ensure_ascii=False, indent=2))
print("AURA_QWEN3_QLORA_COMPLETE", json.dumps(metrics), flush=True)

model.config.use_cache = True
prompt = [{"role":"system","content":"너는 Aura AI다. 정확하고 자연스러운 한국어로 답한다."},{"role":"user","content":"너를 소개하고, 파인튜닝에 사용한 방식을 간단히 설명해 줘."}]
inputs = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to("cuda")
with torch.inference_mode():
    generated = trainer.model.generate(inputs, max_new_tokens=120, do_sample=False)
print("SMOKE_TEST", tokenizer.decode(generated[0][inputs.shape[-1]:], skip_special_tokens=True), flush=True)

import shutil
archive = shutil.make_archive("/kaggle/working/aura-qwen3-0.6b-korean-qlora", "zip", OUTPUT)
print("ADAPTER_ARCHIVE", archive, Path(archive).stat().st_size, flush=True)
